# Whisper large-v3 — zero-shot WER/CER\n\nZero-shots `openai/whisper-large-v3` (no fine-tuning) against every `*test*` split under `data/clean/` (MASC, Casablanca Jordanian, Casablanca Palestinian) and the full `omnilingual_apc` (omni) dataset.\n\nAll heavy lifting lives in `scripts/zero_shot_eval/`:\n- `whisper_predict.py` — `WhisperPredictor`: batched inference, automatic >30s chunking, OOM-safe batch-size backoff.\n- `evaluate.py` — `evaluate(refs, hyps)`: WER/CER after the repo's standard Arabic normalization.\n- `run_eval.py` — `run_dataset(...)`: resumable driver. Predictions are appended to `outputs/whisper_large_v3_zero_shot/<dataset>/predictions.jsonl` and flushed+fsynced after every batch, so **re-running any cell below resumes instead of redoing work** — safe after a kernel crash, OOM, or manual interrupt. Progress (rows/s, GPU memory) is logged every 2 minutes to `outputs/whisper_large_v3_zero_shot/run.log`.

In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
SCRIPTS_DIR = REPO_ROOT / "scripts" / "zero_shot_eval"
sys.path.insert(0, str(SCRIPTS_DIR))

from datasets import discover_eval_targets
from run_eval import run_dataset, setup_logger, DEFAULT_OUT_DIR
from whisper_predict import WhisperPredictConfig, WhisperPredictor

print("repo root:", REPO_ROOT)
print("scripts dir:", SCRIPTS_DIR)

In [ ]:
MODEL_ID = "openai/whisper-large-v3"
BATCH_SIZE = 8          # WhisperPredictor backs this off automatically on CUDA OOM
LIMIT = None            # set to a small int (e.g. 8) for a smoke test, None for the full run
OUT_DIR = DEFAULT_OUT_DIR  # outputs/whisper_large_v3_zero_shot

targets = discover_eval_targets()
for key, files in targets.items():
    print(f"{key}: {len(files)} shard(s)")

logger = setup_logger(OUT_DIR / "run.log")

In [ ]:
# Loads once; safe to re-run (e.g. after a kernel restart) since it just reloads the model.
if "predictor" not in globals():
    predictor = WhisperPredictor(WhisperPredictConfig(model_id=MODEL_ID, batch_size=BATCH_SIZE))
print(predictor.config)

## `test` splits

In [ ]:
key = "casablanca_jordanian"
t0 = time.time()
metrics_path = run_dataset(key, targets[key], predictor, OUT_DIR, BATCH_SIZE, LIMIT, logger)
print(key, "done in", round(time.time() - t0, 1), "s ->", metrics_path)

In [ ]:
key = "casablanca_palestinian"
t0 = time.time()
metrics_path = run_dataset(key, targets[key], predictor, OUT_DIR, BATCH_SIZE, LIMIT, logger)
print(key, "done in", round(time.time() - t0, 1), "s ->", metrics_path)

In [ ]:
key = "masc_c_only"
t0 = time.time()
metrics_path = run_dataset(key, targets[key], predictor, OUT_DIR, BATCH_SIZE, LIMIT, logger)
print(key, "done in", round(time.time() - t0, 1), "s ->", metrics_path)

## Full `omnilingual_apc` (omni) dataset

All 6 shards (`clean` + `recovered_clean`, every split) — durations run up to ~98s, so this is the set that actually exercises the >30s chunking path.

In [ ]:
key = "omnilingual_apc_full"
t0 = time.time()
metrics_path = run_dataset(key, targets[key], predictor, OUT_DIR, BATCH_SIZE, LIMIT, logger)
print(key, "done in", round(time.time() - t0, 1), "s ->", metrics_path)

## Summary

In [ ]:
rows = []
for key in targets:
    mpath = OUT_DIR / key / "metrics.json"
    if not mpath.exists():
        continue
    m = json.loads(mpath.read_text(encoding="utf-8"))
    rows.append({
        "dataset": key,
        "n_total": m["n_total"],
        "n_scored": m["n_scored"],
        "n_dropped_empty_ref": m["n_dropped_empty_ref"],
        "WER_%": None if m["wer"] is None else round(100 * m["wer"], 2),
        "CER_%": None if m["cer"] is None else round(100 * m["cer"], 2),
    })

summary_df = pd.DataFrame(rows)
(OUT_DIR / "summary.json").write_text(
    json.dumps({r["dataset"]: r for r in rows}, ensure_ascii=False, indent=2), encoding="utf-8"
)
summary_df